# LLMSTU tuning — optimize crops, prompts & caption schema
Run this BEFORE the full run. It sweeps crop configs and caption prompts on a small
sample and gives you HTML reports to pick the best settings. Use your **~95GB GPU**
runtime (Runtime → Change runtime type), the same card you'll do the full run on.

Everything runs the model **on the GPU** — no inference API is called.

In [ ]:
REPO_URL = ""  # e.g. https://github.com/WaelK/LLMSTU-pipeline  (see docs/GET_INTO_COLAB.md)
import os
if REPO_URL: !git clone $REPO_URL && cd LLMSTU-pipeline
%cd /content/LLMSTU-pipeline
!pip install -q "ultralytics>=8.3.0" "transformers>=4.57.0" accelerate "bitsandbytes>=0.43.0" \
    "huggingface_hub>=0.35.0" pandas pyarrow pyyaml pillow
from huggingface_hub import login; from google.colab import userdata
login(userdata.get('HF_TOKEN'))

## 1. Grab a small sample of frames to tune on

In [ ]:
from pathlib import Path
from llmstu import dataset_io as io
from llmstu.config import load
cfg = load('config.yaml')
# download just a couple of shards to sample from (fast)
frames_dir = io.download_frames(cfg.data.source_repo, cfg.data.source_subdir,
                                Path('work/frames_dl'), None)
print('frames at', frames_dir)

## 2. Tune CROPS
Edit `experiments.yaml` (crop_variants), then run. Open the HTML it writes to compare
tightness / coverage / crops-per-frame, and copy the winner into `config.yaml` (crop:).

In [ ]:
!python scripts/05_tune_crops.py --frames work/frames_dl --n-frames 8
from IPython.display import HTML
HTML(open('work/tune/crops_report.html').read())

## 3. Make a small crop set for caption tuning
First copy your winning crop settings from step 2 into `config.yaml` (crop:), then run
this — it uses `config.yaml` to make ~30 crops for the caption sweep.

In [ ]:
from llmstu import crop as crop_mod
crop_mod.run(io.iter_frames(frames_dir, cfg.data.frames_glob), Path('work/tune_crops'),
             cfg.crop, manifest_path=Path('work/tune_crops_manifest.jsonl'))

## 4. Tune CAPTIONS (prompt / schema / model)
Edit `experiments.yaml` (caption_variants). These ship with `load_in_4bit: false` to
match your 95GB bf16 production config. Each variant loads its model in turn, so keep
`--n-crops` small (16–32). The report highlights where variants DISAGREE and shows
per-field agreement (agreement = consistency, not correctness — judge correctness by
eye against the thumbnails).

In [ ]:
!python scripts/06_tune_captions.py --crops work/tune_crops --n-crops 24
from IPython.display import HTML
HTML(open('work/tune/captions_report.html').read())

## 5. Lock in your choices
Edit `config.yaml`:
- `crop:` -> winning crop settings
- `caption.prompt_name` -> best prompt, `caption.model_id` -> final model
- adjust `llmstu/schema.py` if you want to add/remove label fields

Then move to the **final run** notebook (`LLMSTU_pipeline_colab.ipynb`) and see
`docs/COLAB_A100.md` for max-efficiency settings.